In [9]:
import os
import cv2
from ultralytics import YOLO
# import ssl  # ssl 모듈 임포트
# import certifi
from sahi.utils.file import download_from_url

# # 1. 윈도우 인증서 저장소 오류를 우회하기 위해 전역 HTTPS 검증을 무력화합니다.
# ssl._create_default_https_context = ssl._create_unverified_context

# # 2. 만약을 위해 requests 환경 변수도 세팅 유지
# os.environ['SSL_CERT_FILE'] = certifi.where()
# os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [10]:
# 작은 차량 이미지 (멀리서 찍은 도로)
download_from_url(
    "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/small-vehicles1.jpeg",
    "demo_data/small-vehicles1.jpeg",
)

# 항공 촬영 지형 이미지
download_from_url(
    "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/terrain2.png",
    "demo_data/terrain2.png",
)

In [11]:
model = YOLO("yolo26s.pt")

In [12]:
input_image_path = "demo_data/small-vehicles1.jpeg"

In [13]:
results = model(input_image_path)


image 1/1 C:\Users\Har29\Desktop\hancom\13_yolo\advenced\demo_data\small-vehicles1.jpeg: 352x640 18 cars, 2 trucks, 117.6ms
Speed: 2.9ms preprocess, 117.6ms inference, 0.2ms postprocess per image at shape (1, 3, 352, 640)


In [14]:
annotated_frame = results[0].plot()

In [15]:
os.makedirs("sahi", exist_ok=True)   # 폴더 없으면 만들고, 있으면 그냥 통과
output_image_path = "sahi/result_org.jpg"
cv2.imwrite(output_image_path, annotated_frame)

True

In [16]:
detected = len(results[0].boxes)

In [17]:
print("=========================")
print(f"기본 YOLO 추론 완료!! {output_image_path}")
print(f"탐지 수: {detected}")

기본 YOLO 추론 완료!! sahi/result_org.jpg
탐지 수: 20


---

# 슬라이싱 추론

---

In [29]:
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

In [30]:
model_path = "yolo26s.pt"

In [35]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.3        # 40% 이상 확신할 때만 탐지
)

In [36]:
results = get_sliced_prediction(
    "demo_data/small-vehicles1.jpeg",
    detection_model,
    slice_height=256,               # 타일 높이 (px)
    slice_width=256,                # 타일 너비 (px)
    overlap_height_ratio=0.2,       # 세로 겹침 10%
    overlap_width_ratio=0.2         # 가로 겹침 10%
)


Performing prediction on 15 slices.


In [37]:
results.export_visuals(export_dir="sahi/")

In [38]:
print(f"탐지 수: {len(results.object_prediction_list)}")
print("모든 코드가 성공적으로 수행됐습니다.")


탐지 수: 22
모든 코드가 성공적으로 수행됐습니다.


---

# 속도 추정

---

In [52]:
from ultralytics import solutions

In [57]:
stream_url = "https://strm1.spatic.go.kr/live/36.stream/chunklist_w626582203.m3u8"
cam = cv2.VideoCapture(stream_url)

In [58]:
yolo_speed = solutions.SpeedEstimator(
    model="yolo11n.pt",            # 사용할 YOLO 모델 파일
    show=False,                    # 솔루션 내부 창 끔 (직접 imshow로 표시)                 # 최대 속도 상한 (km/h) — 상한 초과값은 표시 안 됨
    meter_per_pixel=0.08,           # 픽셀 1개 = 실제 0.5m (환경별 직접 보정 필수)
    classes=[2],                   # COCO class 2 = car (차량만 추적)
    line_width=2,
     fps=15,
    max_hist=10,# 바운딩 박스 선 두께
)

Ultralytics Solutions:  {'source': None, 'model': 'yolo11n.pt', 'classes': [2], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 15, 'max_hist': 10, 'meter_per_pixel': 0.08, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


In [59]:
while cam.isOpened():
    success, frame = cam.read()                # 한 프레임 읽기
    if not success:
        print("프레임 읽기 실패")
        break

    # 3-1. 속도 계산 및 추적 수행 → SolutionResults 반환
    results = yolo_speed(frame)

    # 3-2. 처리된 프레임 표시 (show=False → 우리가 직접 창을 띄움)
    cv2.imshow("SPEED", results.plot_im)

    # 3-3. q 키 입력 시 종료
    if cv2.waitKey(33) & 0xFF == ord('q'):
        print("q 키를 눌러서 종료합니다.")
        break

0: 640x640 0.9ms, 13 car
Speed: 134.5ms track, 0.9ms solution per image at shape (1, 3, 640, 640)

1: 640x640 1.6ms, 12 car
Speed: 118.0ms track, 1.6ms solution per image at shape (1, 3, 640, 640)

2: 640x640 0.8ms, 11 car
Speed: 111.0ms track, 0.8ms solution per image at shape (1, 3, 640, 640)

3: 640x640 1.2ms, 12 car
Speed: 95.6ms track, 1.2ms solution per image at shape (1, 3, 640, 640)

4: 640x640 1.2ms, 12 car
Speed: 88.8ms track, 1.2ms solution per image at shape (1, 3, 640, 640)

5: 640x640 1.3ms, 12 car
Speed: 95.4ms track, 1.3ms solution per image at shape (1, 3, 640, 640)

6: 640x640 0.9ms, 12 car
Speed: 97.0ms track, 0.9ms solution per image at shape (1, 3, 640, 640)

7: 640x640 0.9ms, 12 car
Speed: 87.0ms track, 0.9ms solution per image at shape (1, 3, 640, 640)

8: 640x640 1.3ms, 12 car
Speed: 88.3ms track, 1.3ms solution per image at shape (1, 3, 640, 640)

9: 640x640 2.4ms, 12 car
Speed: 95.7ms track, 2.4ms solution per image at shape (1, 3, 640, 640)

10: 640x640 2.1ms

In [60]:
cam.release()
cv2.destroyAllWindows()

---

# SAM — Segment Anything Model

---

In [61]:
from ultralytics import SAM
import time

In [62]:
model = SAM("sam_b.pt")
print("SAM 모델 로드 완료!!")

SAM 모델 로드 완료!!


In [63]:
image_path = "demo_data/small-vehicles1.jpeg"

In [64]:
start_time = time.time()

In [ ]:
model(image_path, save=True)

In [ ]:
end_time = time.time()

---

# OpenVINO

---

In [1]:
from ultralytics import YOLO

In [2]:
model = YOLO("yolo26s.pt")

In [3]:
model.export(format="openvino")

Ultralytics 8.4.103  Python-3.11.15 torch-2.13.0+cpu CPU (Intel Core i7-8700 3.20GHz)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,496,140 parameters, 0 gradients, 20.7 GFLOPs

PyTorch: starting from 'yolo26s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.5 MB)

OpenVINO: starting export with openvino 2026.2.1-21919-ede283a88e3-releases/2026/2...
OpenVINO: export success  4.7s, saved as 'yolo26s_openvino_model\' (36.7 MB)

Export complete (5.7s)
Results saved to C:\Users\Har29\Desktop\hancom\13_yolo\advenced\yolo26s_openvino_model
Predict:         yolo predict task=detect model=yolo26s_openvino_model\ imgsz=640 
Validate:        yolo val task=detect model=yolo26s_openvino_model\ imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26s_openvino_model\\'